# LeetCode 2839 — Check if Strings Can be Made Equal With Operations I bằng Python

Link đề: https://leetcode.com/problems/check-if-strings-can-be-made-equal-with-operations-i/ · Độ khó: **Easy** · Chủ đề: String, Sorting, Swap

Skeleton LeetCode:

```python
class Solution:
    def canBeEqual(self, s1: str, s2: str) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai chuỗi `s1`, `s2` **độ dài 4**; một **thao tác** cho phép chọn hai chỉ số `i < j` với `j − i = 2` rồi **hoán đổi** hai ký tự đó trong **một** trong hai chuỗi; hỏi có thể làm hai chuỗi **bằng nhau** không.
2. Biết **tự tính tay**: chỉ có hai kiểu hoán đổi `(0, 2)` và `(1, 3)`, nên các vị trí **chẵn** (0, 2) và **lẻ** (1, 3) **không bao giờ lẫn nhau**.
3. Học các công cụ Python: chỉ số chuỗi, **cắt lát có bước** `s[0::2]`, `sorted`, nối `+`, `set`, `deque` / hàng đợi cho BFS, tổ hợp `itertools.product`.
4. Đi từ **Brute Force: BFS qua mọi chuỗi có thể đạt từ `s1`** → **thử 4 tổ hợp hoán đổi (mỗi phép dùng 0 hoặc 1 lần)** → **so sánh đa tập hai lớp chẵn/lẻ bằng `sorted`** → **điều kiện so sánh trực tiếp**.
5. Chứng minh vì sao đúng ("hoán đổi hai lần cùng cặp thì quay về; hai phép giao hoán") và tránh lỗi kinh điển: so sánh cả chuỗi sau `sorted`, hoán đổi các cặp kề nhau, quên rằng vị trí chẵn / lẻ độc lập.

## 2. Đề bài & ví dụ

Bạn được cho hai chuỗi `s1` và `s2`, **mỗi chuỗi có độ dài đúng 4**, gồm chữ cái thường.

Bạn có thể thực hiện thao tác sau **trên bất kỳ chuỗi nào, bao nhiêu lần tuỳ ý**:

- Chọn hai chỉ số `i` và `j` sao cho `j − i = 2`, rồi **hoán đổi** hai ký tự tại các chỉ số đó trong chuỗi.

Trả về `true` nếu bạn có thể làm cho `s1` và `s2` **bằng nhau**, ngược lại trả về `false`.

Vì độ dài là 4 và `j − i = 2`, chỉ có hai cặp chỉ số hợp lệ: `(0, 2)` và `(1, 3)`.

| Ví dụ | `s1`, `s2` | Output | Giải thích |
|---|---|---|---|
| 1 | `"abcd"`, `"cdab"` | `true` | Chọn `i = 0, j = 2` trên `s1`: `"cbad"`. Rồi chọn `i = 1, j = 3` trên `"cbad"`: `"cdab"` = `s2`. |
| 2 | `"abcd"`, `"dacb"` | `false` | Không thể làm hai chuỗi bằng nhau (chữ `d` của `s2` nằm ở vị trí 0, mà vị trí 0 chỉ nhận các chữ từ vị trí 0 hoặc 2 của `s1`, tức `a` hoặc `c`). |

Constraints:

- `s1.length == s2.length == 4`
- `s1` và `s2` chỉ gồm chữ cái thường tiếng Anh.

In [1]:
# Chạy thử "bằng tay" ví dụ 1: hai phép hoán đổi
s = list("abcd")
s[0], s[2] = s[2], s[0]          # hoán đổi chỉ số 0 và 2
print("".join(s))                # cbad
s[1], s[3] = s[3], s[1]          # hoán đổi chỉ số 1 và 3
print("".join(s))                # cdab

cbad
cdab


## 3. Kiến thức nền

### 3.1. Chỉ số chuỗi và chuỗi bất biến

Chuỗi `"abcd"` có chỉ số `0, 1, 2, 3`: `s[0] = 'a'`, `s[3] = 'd'`. Chuỗi **bất biến**, không gán được `s[0] = 'c'`. Để hoán đổi, ta đổi sang `list` ký tự, hoán đổi, rồi `"".join(...)` lại.

In [2]:
s = "abcd"
print(s[0], s[1], s[2], s[3])      # a b c d
try:
    s[0] = "z"
except TypeError as e:
    print("lỗi:", e)               # 'str' object does not support item assignment

a b c d
lỗi: 'str' object does not support item assignment


### 3.2. Hoán đổi hai phần tử: `a[i], a[j] = a[j], a[i]`

Python cho phép gán đồng thời. Vế phải được tính xong **trước**, rồi mới gán, nên không cần biến tạm.

In [3]:
a = list("abcd")
a[0], a[2] = a[2], a[0]
print(a)             # ['c', 'b', 'a', 'd']
a[0], a[2] = a[2], a[0]          # hoán đổi lần nữa: quay về như cũ
print(a)             # ['a', 'b', 'c', 'd']

['c', 'b', 'a', 'd']
['a', 'b', 'c', 'd']


### 3.3. Cắt lát có bước `s[start::step]`

`s[0::2]` lấy các ký tự ở chỉ số **chẵn** `0, 2, ...`; `s[1::2]` lấy chỉ số **lẻ** `1, 3, ...`. Với chuỗi độ dài 4: `s[0::2] = s[0] + s[2]`, `s[1::2] = s[1] + s[3]`.

In [4]:
s = "abcd"
print(s[0::2])       # ac  (chỉ số chẵn: 0, 2)
print(s[1::2])       # bd  (chỉ số lẻ: 1, 3)
print(s[::2], s[::-1])   # ac dcba

ac
bd
ac dcba


### 3.4. `sorted` trên chuỗi và so sánh đa tập

`sorted("ca")` cho `['a', 'c']`: danh sách ký tự đã sắp xếp. Hai chuỗi có **cùng các ký tự với cùng số lần** (cùng "đa tập") khi và chỉ khi `sorted` của chúng bằng nhau. Dùng để hỏi "hai chuỗi có phải hoán vị của nhau không".

In [5]:
print(sorted("ca"), sorted("ac"))              # ['a', 'c'] ['a', 'c']
print(sorted("ca") == sorted("ac"))            # True
print(sorted("aab") == sorted("abb"))          # False

['a', 'c'] ['a', 'c']
True
False


### 3.5. `set` các trạng thái và BFS cơ bản

Để liệt kê **mọi chuỗi có thể đạt**, ta giữ một `set` các chuỗi đã gặp và một hàng đợi (`deque`) các chuỗi cần mở rộng. Mỗi lần lấy một chuỗi, thử hai phép hoán đổi, chuỗi mới chưa gặp thì thêm vào. Kỹ thuật này gọi là **BFS (tìm kiếm theo chiều rộng)**.

In [6]:
from collections import deque

seen = {"abcd"}
queue = deque(["abcd"])
print(seen, list(queue))
cur = queue.popleft()
print("lấy ra:", cur, "| hàng đợi còn:", list(queue))

{'abcd'} ['abcd']
lấy ra: abcd | hàng đợi còn: []


### 3.6. `itertools.product` — thử mọi tổ hợp bật/tắt

`product([0, 1], repeat=2)` cho các cặp `(0,0), (0,1), (1,0), (1,1)`: mỗi phần tử là "có dùng phép hoán đổi thứ 1 / thứ 2 hay không".

In [7]:
from itertools import product
print(list(product([0, 1], repeat=2)))     # [(0, 0), (0, 1), (1, 0), (1, 1)]

[(0, 0), (0, 1), (1, 0), (1, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy ghi các **vị trí** của chuỗi độ dài 4:

```
vị trí:   0   1   2   3
          a   b   c   d
          └───2───┘          (cặp 0-2)
              └───2───┘      (cặp 1-3)
```

Hai phép hoán đổi:

- Phép A: hoán đổi vị trí `0` và `2`.
- Phép B: hoán đổi vị trí `1` và `3`.

**Nhận xét quan trọng 1:** phép A chỉ đụng vị trí `0, 2`; phép B chỉ đụng vị trí `1, 3`. Hai nhóm vị trí `{0, 2}` và `{1, 3}` **không bao giờ trao đổi ký tự với nhau**. Một ký tự ở vị trí chẵn luôn ở vị trí chẵn.

**Nhận xét quan trọng 2:** hoán đổi một cặp hai lần thì quay về như cũ; hai phép A và B không ảnh hưởng nhau nên làm theo thứ tự nào cũng được. Do đó mỗi phép chỉ cần xét "dùng **0** hoặc **1** lần": tối đa **4** chuỗi khác nhau từ một chuỗi: `abcd`, `cbad` (A), `adcb` (B), `cdab` (A và B).

**Hệ quả:** `s1` biến thành `s2` được khi và chỉ khi:

- (hai ký tự ở vị trí chẵn của `s1`) có thể xếp thành (hai ký tự ở vị trí chẵn của `s2`), tức **cùng đa tập**; **và**
- (hai ký tự ở vị trí lẻ của `s1`) cùng đa tập với (hai ký tự ở vị trí lẻ của `s2`).

Vì sao cho cả hai chuỗi hoán đổi cũng không thêm khả năng? Một thao tác trên `s2` có thể "dịch ngược" thành thao tác trên `s1` (vì hoán đổi tự đảo ngược), nên chỉ cần xét biến đổi `s1` thành `s2` (sẽ chứng minh kĩ ở mục 10).

In [8]:
s1, s2 = "abcd", "cdab"
print("chẵn s1:", s1[0::2], "| chẵn s2:", s2[0::2], "| cùng đa tập:", sorted(s1[0::2]) == sorted(s2[0::2]))
print("lẻ   s1:", s1[1::2], "| lẻ   s2:", s2[1::2], "| cùng đa tập:", sorted(s1[1::2]) == sorted(s2[1::2]))
s2 = "dacb"
print("---")
print("chẵn s1:", s1[0::2], "| chẵn s2:", s2[0::2], "| cùng đa tập:", sorted(s1[0::2]) == sorted(s2[0::2]))

chẵn s1: ac | chẵn s2: ca | cùng đa tập: True
lẻ   s1: bd | lẻ   s2: db | cùng đa tập: True
---
chẵn s1: ac | chẵn s2: dc | cùng đa tập: False


## 5. Cách 1 — Brute Force / baseline: BFS qua mọi chuỗi có thể đạt từ `s1`

**Ý tưởng:** liệt kê **mọi chuỗi** mà `s1` có thể biến thành bằng cách áp dụng các phép hoán đổi nhiều lần (BFS). Nếu `s2` nằm trong tập đó thì `true`.

Vì ta được hoán đổi trên **cả hai chuỗi**, nhưng hoán đổi tự đảo ngược, nên "`s1` và `s2` có thể bằng nhau" ⇔ "`s2` đạt được từ `s1`" (ta sẽ chứng minh ở mục 10). Cách này dịch thẳng đề, rất tốt làm **oracle**.

In [9]:
from collections import deque

def swap(s: str, i: int, j: int) -> str:
    a = list(s)
    a[i], a[j] = a[j], a[i]
    return "".join(a)

def can_be_equal_bfs(s1: str, s2: str) -> bool:
    seen = {s1}
    queue = deque([s1])
    while queue:
        cur = queue.popleft()
        if cur == s2:
            return True
        for i, j in ((0, 2), (1, 3)):
            nxt = swap(cur, i, j)
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return False

In [10]:
print(can_be_equal_bfs("abcd", "cdab"))     # True
print(can_be_equal_bfs("abcd", "dacb"))     # False
print(can_be_equal_bfs("abcd", "abcd"))     # True (đã bằng nhau, 0 thao tác)

True
False
True


**Độ phức tạp:** từ một chuỗi đạt được tối đa `4` chuỗi (như đã phân tích), mỗi chuỗi thử `2` phép, mỗi phép tạo chuỗi độ dài 4 ⇒ Time `O(1)` (hằng số), Space `O(1)`. Tuy nhiên đây là hằng số "ẩn" của BFS; nếu tổng quát hoá độ dài chuỗi thì số trạng thái bùng nổ.

**Nút thắt:** dù nhanh với độ dài 4, BFS thể hiện **hơi thừa**: ta không cần liệt kê trạng thái, chỉ cần biết hai lớp vị trí độc lập.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — chỉ có 4 trạng thái.** Vì mỗi phép dùng 0 hoặc 1 lần và hai phép giao hoán, từ `s1` đạt được đúng các chuỗi ứng với `(a, b) ∈ {0, 1}²`: dùng phép A hay không, dùng phép B hay không. Thử cả 4 và so với `s2`.

**Đột phá 2 — hai lớp vị trí độc lập.** Hai ký tự vị trí chẵn của `s2` phải là hoán vị của hai ký tự vị trí chẵn của `s1`; tương tự với vị trí lẻ. So sánh `sorted(s1[0::2]) == sorted(s2[0::2])` và `sorted(s1[1::2]) == sorted(s2[1::2])`.

**Đột phá 3 — viết ra điều kiện trực tiếp.** Với hai ký tự: "hoán vị của nhau" ⇔ `(x1 == y1 và x2 == y2)` hoặc `(x1 == y2 và x2 == y1)`.

In [11]:
s1, s2 = "abcd", "cdab"
for a, b in product([0, 1], repeat=2):
    t = s1
    if a:
        t = swap(t, 0, 2)
    if b:
        t = swap(t, 1, 3)
    print(f"dùng A={a}, B={b} -> {t}", "<- bằng s2" if t == s2 else "")

dùng A=0, B=0 -> abcd 
dùng A=0, B=1 -> adcb 
dùng A=1, B=0 -> cbad 
dùng A=1, B=1 -> cdab <- bằng s2


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — Thử 4 tổ hợp hoán đổi

**Từng đoạn:**

- Với `a` trong `{0, 1}` (dùng phép `(0, 2)` hay không) và `b` trong `{0, 1}` (dùng phép `(1, 3)` hay không): tạo chuỗi `t` từ `s1`.
- Nếu `t == s2` ở bất kỳ tổ hợp nào ⇒ `True`.
- Không tổ hợp nào ⇒ `False`.

In [12]:
def can_be_equal_combos(s1: str, s2: str) -> bool:
    for a, b in product([0, 1], repeat=2):
        t = s1
        if a:
            t = swap(t, 0, 2)
        if b:
            t = swap(t, 1, 3)
        if t == s2:
            return True
    return False

In [13]:
print(can_be_equal_combos("abcd", "cdab"))     # True
print(can_be_equal_combos("abcd", "dacb"))     # False

True
False


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def canBeEqual(self, s1: str, s2: str) -> bool:
        def swap(s, i, j):
            a = list(s)
            a[i], a[j] = a[j], a[i]
            return "".join(a)

        for a in (0, 1):
            for b in (0, 1):
                t = s1
                if a:
                    t = swap(t, 0, 2)
                if b:
                    t = swap(t, 1, 3)
                if t == s2:
                    return True
        return False
```

**Độ phức tạp:** 4 tổ hợp, mỗi tổ hợp thao tác trên chuỗi độ dài 4 ⇒ Time `O(1)`, Space `O(1)`.

### Cách 3 — So sánh đa tập hai lớp chẵn / lẻ bằng `sorted`

**Từng đoạn:**

- `s1[0::2]`, `s2[0::2]`: hai ký tự ở chỉ số chẵn mỗi chuỗi; `sorted` cho dạng chuẩn.
- `s1[1::2]`, `s2[1::2]`: hai ký tự ở chỉ số lẻ.
- Trả về `True` nếu **cả hai** cặp đều bằng nhau sau `sorted`.

In [14]:
def can_be_equal_sorted(s1: str, s2: str) -> bool:
    return (sorted(s1[0::2]) == sorted(s2[0::2])
            and sorted(s1[1::2]) == sorted(s2[1::2]))

In [15]:
print(can_be_equal_sorted("abcd", "cdab"))     # True
print(can_be_equal_sorted("abcd", "dacb"))     # False

True
False


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def canBeEqual(self, s1: str, s2: str) -> bool:
        return (sorted(s1[0::2]) == sorted(s2[0::2])
                and sorted(s1[1::2]) == sorted(s2[1::2]))
```

**Độ phức tạp:** Time `O(1)` (độ dài cố định 4; tổng quát `O(n log n)`), Space `O(1)`.

### Cách 4 — Điều kiện so sánh trực tiếp (không `sorted`)

**Từng đoạn:** hai ký tự `(x1, x2)` là hoán vị của `(y1, y2)` ⇔ `(x1 == y1 and x2 == y2) or (x1 == y2 and x2 == y1)`. Áp dụng cho cặp chẵn `(s1[0], s1[2])` với `(s2[0], s2[2])`, và cặp lẻ `(s1[1], s1[3])` với `(s2[1], s2[3])`.

In [16]:
def can_be_equal_direct(s1: str, s2: str) -> bool:
    even_ok = (s1[0] == s2[0] and s1[2] == s2[2]) or (s1[0] == s2[2] and s1[2] == s2[0])
    odd_ok = (s1[1] == s2[1] and s1[3] == s2[3]) or (s1[1] == s2[3] and s1[3] == s2[1])
    return even_ok and odd_ok

In [17]:
print(can_be_equal_direct("abcd", "cdab"))     # True
print(can_be_equal_direct("abcd", "dacb"))     # False

True
False


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def canBeEqual(self, s1: str, s2: str) -> bool:
        even_ok = (s1[0] == s2[0] and s1[2] == s2[2]) or (s1[0] == s2[2] and s1[2] == s2[0])
        odd_ok = (s1[1] == s2[1] and s1[3] == s2[3]) or (s1[1] == s2[3] and s1[3] == s2[1])
        return even_ok and odd_ok
```

**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 (thử 4 tổ hợp) với `s1 = "abcd"`, `s2 = "cdab"`; và trace Cách 3 để thấy hai lớp chẵn / lẻ.

| `a` (phép 0–2) | `b` (phép 1–3) | chuỗi `t` | `t == s2`? |
|---|---|---|---|
| 0 | 0 | `abcd` | không |
| 0 | 1 | `adcb` | không |
| 1 | 0 | `cbad` | không |
| 1 | 1 | `cdab` | **có** |

In [18]:
def trace_combos(s1, s2):
    print(f"s1={s1}, s2={s2}")
    for a, b in product([0, 1], repeat=2):
        t = s1
        if a:
            t = swap(t, 0, 2)
        if b:
            t = swap(t, 1, 3)
        print(f"  a={a}, b={b}: t={t}  {'<- bằng s2' if t == s2 else ''}")
        if t == s2:
            return True
    return False

print(trace_combos("abcd", "cdab"))
print(trace_combos("abcd", "dacb"))

s1=abcd, s2=cdab
  a=0, b=0: t=abcd  
  a=0, b=1: t=adcb  
  a=1, b=0: t=cbad  
  a=1, b=1: t=cdab  <- bằng s2
True
s1=abcd, s2=dacb
  a=0, b=0: t=abcd  
  a=0, b=1: t=adcb  
  a=1, b=0: t=cbad  
  a=1, b=1: t=cdab  
False


In [19]:
def trace_parity(s1, s2):
    for ten, sl in (("chẵn (0,2)", slice(0, None, 2)), ("lẻ (1,3)", slice(1, None, 2))):
        a, b = s1[sl], s2[sl]
        print(f"lớp {ten}: s1 -> {a!r} sorted {sorted(a)} | s2 -> {b!r} sorted {sorted(b)} | bằng nhau: {sorted(a) == sorted(b)}")
    return can_be_equal_sorted(s1, s2)

print(trace_parity("abcd", "cdab"))
print("---")
print(trace_parity("abcd", "dacb"))

lớp chẵn (0,2): s1 -> 'ac' sorted ['a', 'c'] | s2 -> 'ca' sorted ['a', 'c'] | bằng nhau: True
lớp lẻ (1,3): s1 -> 'bd' sorted ['b', 'd'] | s2 -> 'db' sorted ['b', 'd'] | bằng nhau: True
True
---
lớp chẵn (0,2): s1 -> 'ac' sorted ['a', 'c'] | s2 -> 'dc' sorted ['c', 'd'] | bằng nhau: False
lớp lẻ (1,3): s1 -> 'bd' sorted ['b', 'd'] | s2 -> 'ab' sorted ['a', 'b'] | bằng nhau: False
False


## 9. Code nộp LeetCode

Dùng Cách 3 (so sánh đa tập hai lớp chẵn / lẻ).

In [20]:
class Solution:
    def canBeEqual(self, s1: str, s2: str) -> bool:
        # vị trí chẵn {0, 2} và vị trí lẻ {1, 3} hoán đổi độc lập
        return (sorted(s1[0::2]) == sorted(s2[0::2])
                and sorted(s1[1::2]) == sorted(s2[1::2]))

In [21]:
print(Solution().canBeEqual("abcd", "cdab"))     # True
print(Solution().canBeEqual("abcd", "dacb"))     # False

True
False


## 10. Vì sao thuật toán đúng

**Bước 1 — mô hình hoá.** Một chuỗi độ dài 4 là bộ `(p0, p1, p2, p3)`. Phép A đổi chỗ `p0, p2`; phép B đổi chỗ `p1, p3`. Mỗi phép là một **hoán vị** của các vị trí; cả hai đều là **đảo ngược chính nó** (áp dụng hai lần = không đổi) và **giao hoán** (A rồi B = B rồi A, vì chúng đụng hai tập vị trí rời nhau).

**Bước 2 — tập chuỗi đạt được từ `s1`.** Bất kỳ dãy phép nào cũng rút gọn thành "A dùng 0 hoặc 1 lần, B dùng 0 hoặc 1 lần" (vì giao hoán và tự đảo ngược). Vậy từ `s1` đạt được đúng 4 chuỗi (có thể trùng nhau): `s1`, `A(s1)`, `B(s1)`, `A(B(s1))`.

**Bước 3 — thao tác trên cả hai chuỗi không thêm khả năng.** Nếu ta dùng dãy phép `P` trên `s1` và dãy `Q` trên `s2` để chúng bằng nhau: `P(s1) = Q(s2)`. Áp dụng ngược `Q` (vì mọi phép tự đảo ngược và giao hoán, nghịch đảo của `Q` cũng là một dãy phép trong nhóm) vào hai vế: `Q⁻¹(P(s1)) = s2`. Tức `s2` đạt được từ `s1` chỉ bằng thao tác trên `s1`. Ngược lại, nếu `s2` đạt được từ `s1`, ta chỉ cần thao tác trên `s1`.

**Bước 4 — điều kiện hoán vị theo lớp.**

- Phép A chỉ làm thay đổi cặp `(p0, p2)` giữa hai thứ tự `(x, y)` và `(y, x)`; `p1, p3` giữ nguyên. Phép B tương tự cho `(p1, p3)`.
- Vậy `s2` đạt được từ `s1` ⇔ cặp `(s2[0], s2[2])` bằng `(s1[0], s1[2])` hoặc đảo của nó **và** cặp `(s2[1], s2[3])` bằng `(s1[1], s1[3])` hoặc đảo của nó.
- Hai cặp `(x, y)` và `(u, v)` bằng nhau hoặc đảo nhau ⇔ `sorted([x, y]) == sorted([u, v])` ⇔ điều kiện của Cách 3 và Cách 4.

Cả bốn cách tính cùng một đại lượng. Ta cũng kiểm bằng **vét cạn** trên các bảng chữ nhỏ ở mục 12.

## 11. Phân tích Time / Space Complexity

Độ dài chuỗi cố định `n = 4`, nên mọi cách là `O(1)`. Nếu muốn nhìn ở dạng tổng quát (độ dài `n` với hoán đổi khoảng cách 2), ta có:

| Cách | Time (n = 4) | Space | Ghi chú |
|---|---|---|---|
| 1. BFS mọi chuỗi đạt được | `O(1)` (≤ 4 trạng thái) | `O(1)` | oracle; tổng quát bùng nổ theo `n` |
| 2. Thử 4 tổ hợp | `O(1)` | `O(1)` | dựa vào "mỗi phép dùng 0/1 lần" |
| 3. So `sorted` chẵn / lẻ | `O(1)` | `O(1)` | tổng quát `O(n log n)` (chẵn lẻ độc lập) |
| 4. So sánh trực tiếp | `O(1)` | `O(1)` | nhanh nhất, ít linh hoạt |

Ở dạng tổng quát (xem LeetCode 2840), cách 3 vẫn đúng: hoán đổi khoảng cách 2 cho phép **hoán vị tuỳ ý** các ký tự cùng tính chẵn lẻ vị trí.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: giống hệt nhau; chỉ cần phép A; chỉ cần phép B; cần cả hai; khác nhau ở một ký tự; cùng ký tự nhưng sai tính chẵn lẻ; chuỗi có ký tự lặp.

In [22]:
impls = {
    "combos": can_be_equal_combos,
    "sorted": can_be_equal_sorted,
    "direct": can_be_equal_direct,
    "Solution": Solution().canBeEqual,
}
cases = [
    ("ví dụ 1", "abcd", "cdab", True),
    ("ví dụ 2", "abcd", "dacb", False),
    ("giống hệt", "abcd", "abcd", True),
    ("chỉ phép A (0-2)", "abcd", "cbad", True),
    ("chỉ phép B (1-3)", "abcd", "adcb", True),
    ("khác một ký tự", "abcd", "abce", False),
    ("cùng ký tự sai tính chẵn lẻ", "abcd", "badc", False),
    ("đảo kề nhau không được", "abcd", "bacd", False),
    ("ký tự lặp, cần cả A và B", "aabb", "bbaa", True),
    ("ký tự lặp, sai tính chẵn lẻ", "abab", "baba", False),
    ("toàn giống nhau", "aaaa", "aaaa", True),
    ("hai lớp cùng ký tự", "abab", "abab", True),
]
for name, s1, s2, exp in cases:
    assert can_be_equal_bfs(s1, s2) == exp, ("bfs", name)
    for k, f in impls.items():
        got = f(s1, s2)
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> True
PASS ví dụ 2 -> False
PASS giống hệt -> True
PASS chỉ phép A (0-2) -> True
PASS chỉ phép B (1-3) -> True
PASS khác một ký tự -> False
PASS cùng ký tự sai tính chẵn lẻ -> False
PASS đảo kề nhau không được -> False
PASS ký tự lặp, cần cả A và B -> True
PASS ký tự lặp, sai tính chẵn lẻ -> False
PASS toàn giống nhau -> True
PASS hai lớp cùng ký tự -> True


Kiểm tay vài ca: `"abcd"` → `"badc"`: vị trí 0 của `s2` là `b`, mà `b` nằm ở vị trí lẻ của `s1`; không thể sang vị trí chẵn ⇒ `False` ✓. `"aabb"` → `"bbaa"`: hoán đổi `(0, 2)` cho `"baab"`, rồi hoán đổi `(1, 3)` cho `"bbaa"` ⇒ `True` ✓. `"abab"` → `"baba"`: lớp chẵn của `s1` là `{a, a}` nhưng của `s2` là `{b, b}` ⇒ `False` ✓.

### 12.2. Vét cạn trên bảng chữ nhỏ

Với bảng chữ `{a, b, c}` có `3^4 = 81` chuỗi độ dài 4; cặp `(s1, s2)` có `81 × 81 = 6561`. Ta kiểm **tất cả** các cặp bằng cả bốn cách và so với BFS (oracle).

In [23]:
alpha = "abc"
all_strings = ["".join(p) for p in product(alpha, repeat=4)]
cnt_true = 0
for s1 in all_strings:
    for s2 in all_strings:
        exp = can_be_equal_bfs(s1, s2)
        cnt_true += exp
        for k, f in impls.items():
            assert f(s1, s2) == exp, (s1, s2, k)
print(len(all_strings), "chuỗi;", len(all_strings) ** 2, "cặp: mọi cách khớp BFS; số cặp True =", cnt_true)

81 chuỗi; 6561 cặp: mọi cách khớp BFS; số cặp True = 225


### 12.3. Ngẫu nhiên có seed và tính chất

Input hợp lệ theo constraints: độ dài 4, chữ thường. Ta sinh `s2` bằng cách áp dụng ngẫu nhiên các phép hoán đổi cho `s1` (kết quả luôn `True`), và sinh cặp hoàn toàn ngẫu nhiên.

In [24]:
import random
random.seed(2839)
letters = "abcdefghijklmnopqrstuvwxyz"
for _ in range(2000):
    s1 = "".join(random.choice(letters[:random.choice([2, 3, 26])]) for _ in range(4))
    # s2 đạt được từ s1 bằng các hoán đổi ngẫu nhiên: luôn True
    s2 = s1
    for _ in range(random.randint(0, 6)):
        i, j = random.choice([(0, 2), (1, 3)])
        s2 = swap(s2, i, j)
    assert Solution().canBeEqual(s1, s2) is True
    # cặp ngẫu nhiên hoàn toàn: so với BFS
    t1 = "".join(random.choice("abc") for _ in range(4))
    t2 = "".join(random.choice("abc") for _ in range(4))
    assert Solution().canBeEqual(t1, t2) == can_be_equal_bfs(t1, t2)
    # tính đối xứng
    assert Solution().canBeEqual(t1, t2) == Solution().canBeEqual(t2, t1)
print("ngẫu nhiên: đúng; tính đối xứng đúng")

ngẫu nhiên: đúng; tính đối xứng đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: So sánh `sorted` cả chuỗi (bỏ qua tính chẵn / lẻ)

In [25]:
def sai_sorted_ca_chuoi(s1, s2):
    return sorted(s1) == sorted(s2)       # sai: cho phép hoán vị tuỳ ý

print("sai :", sai_sorted_ca_chuoi("abcd", "dacb"))           # True (sai)
print("đúng:", Solution().canBeEqual("abcd", "dacb"))          # False

sai : True
đúng: False


**Vì sao sai:** hai chuỗi có cùng chữ cái chưa đủ; ký tự chỉ được **dời qua lại giữa các vị trí cùng tính chẵn / lẻ**. Ví dụ 2 của đề chính là bẫy này.

### Lỗi 2: Hoán đổi các vị trí kề nhau

In [26]:
def sai_hoan_doi_ke(s1, s2):
    for a, b in product([0, 1], repeat=2):
        t = s1
        if a:
            t = swap(t, 0, 1)      # sai: đề cho khoảng cách 2, không phải 1
        if b:
            t = swap(t, 2, 3)
        if t == s2:
            return True
    return False

print("sai :", sai_hoan_doi_ke("abcd", "bacd"))           # True (sai)
print("đúng:", Solution().canBeEqual("abcd", "bacd"))      # False

sai : True
đúng: False


**Vì sao sai:** đề chỉ cho hoán đổi khi `j − i = 2`. Hoán đổi `(0, 1)` hay `(2, 3)` không được phép.

### Lỗi 3: Chỉ dùng mỗi phép một lần theo đúng thứ tự A rồi B và bỏ qua trường hợp "chỉ B"

In [27]:
def sai_thieu_to_hop(s1, s2):
    t = swap(s1, 0, 2)               # sai: bắt buộc dùng phép A
    t = swap(t, 1, 3)                # sai: bắt buộc dùng phép B
    return t == s2 or s1 == s2

print("sai :", sai_thieu_to_hop("abcd", "adcb"))          # False (sai, chỉ cần phép B)
print("đúng:", Solution().canBeEqual("abcd", "adcb"))      # True

sai : False
đúng: True


**Vì sao sai:** mỗi phép có thể dùng hoặc **không**: có 4 tổ hợp `(0,0), (0,1), (1,0), (1,1)`. Bỏ sót tổ hợp thì bỏ sót đáp án.

### Lỗi 4: Nghĩ phải hoán đổi cả `s2`
Nhiều người băn khoăn vì đề cho hoán đổi ở **một trong hai chuỗi**. Như đã chứng minh, chỉ cần thao tác trên một chuỗi là đủ; không cần đụng tới `s2`.

In [28]:
# s2 đạt được từ s1 <=> s1 đạt được từ s2 (đối xứng)
for s1, s2 in [("abcd", "cdab"), ("abcd", "dacb"), ("aabb", "bbaa")]:
    assert Solution().canBeEqual(s1, s2) == Solution().canBeEqual(s2, s1)
print("tính đối xứng đúng")

tính đối xứng đúng


### Lỗi 5 (nhỏ): Sửa chuỗi tại chỗ

`s1[0] = s1[2]` gây `TypeError` vì chuỗi bất biến. Hãy chuyển sang `list` rồi `"".join`.

In [29]:
s = "abcd"
a = list(s)
a[0], a[2] = a[2], a[0]
s = "".join(a)
print(s)     # cbad

cbad


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "hoán đổi hai vị trí cách nhau `k`" ⇒ các vị trí chia thành **lớp** theo `i mod k`; trong mỗi lớp được hoán vị **tuỳ ý**, giữa các lớp thì **không**.
- Hai chuỗi có thể biến thành nhau ⇔ với mỗi lớp, **đa tập ký tự** bằng nhau ⇒ dùng `sorted` hoặc `Counter` trên từng lớp.
- Với độ dài rất nhỏ (4), có thể **vét cạn** mọi trạng thái bằng BFS / tổ hợp để làm oracle.
- Phép hoán đổi **tự đảo ngược** và các phép rời nhau **giao hoán**: chỉ cần xét 0 hoặc 1 lần mỗi phép.
- Cắt lát `s[0::2]`, `s[1::2]` là cách gọn nhất để lấy lớp chẵn / lẻ.

## 15. Biến thể của bài

1. **Độ dài bất kỳ `n`** (LeetCode 2840): hoán đổi `j − i = 2` trên chuỗi dài; so sánh `sorted(s1[0::2]) == sorted(s2[0::2])` và `sorted(s1[1::2]) == sorted(s2[1::2])`.
2. **Hoán đổi khoảng cách `k` bất kỳ:** chia thành `k` lớp theo `i % k`.
3. **Hoán đổi kề nhau** (`j − i = 1`): hoán vị tuỳ ý, chỉ cần `sorted(s1) == sorted(s2)`.
4. **Số thao tác tối thiểu** để biến `s1` thành `s2`: tối đa 2 ở bài này (phép A và/hoặc B).
5. **Trả về dãy thao tác** thay vì chỉ có / không.
6. **Hoán đổi chỉ trên `s1`:** như đã chứng minh, kết quả không đổi.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | liệt kê mọi chuỗi đạt được | `O(1)` (≤ 4 trạng thái) | `O(1)` | oracle |
| 2. Thử 4 tổ hợp | mỗi phép dùng 0 hoặc 1 lần | `O(1)` | `O(1)` | khi quen "bật / tắt" |
| 3. `sorted` chẵn / lẻ | hai lớp độc lập, so đa tập | `O(1)` | `O(1)` | **code nộp**, mở rộng được cho độ dài `n` |
| 4. So sánh trực tiếp | điều kiện `(x1==y1 and ...) or (...)` | `O(1)` | `O(1)` | nhanh nhất cho `n = 4` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"abcd"[0::2]`, `"abcd"[1::2]`, `sorted("ca")`, `swap("abcd", 1, 3)` (dùng hàm `swap` ở trên).

**Bài 2 (tính tay).** Với `s1 = "abcd"` và `s2 = "cbad"`, `s2 = "dcba"`, `s2 = "badc"`: quyết định `True` / `False` bằng cách xét lớp chẵn / lẻ.

**Bài 3 (điền TODO).** Hoàn thành `can_be_equal_todo(s1, s2)` theo Cách 3.

In [30]:
def can_be_equal_todo(s1, s2):
    # TODO 1: lấy lớp chẵn của s1 và s2 bằng cắt lát [0::2]
    # TODO 2: lấy lớp lẻ của s1 và s2 bằng cắt lát [1::2]
    # TODO 3: so sánh sorted của lớp chẵn
    # TODO 4: so sánh sorted của lớp lẻ
    # TODO 5: trả về True nếu cả hai đều bằng nhau
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `can_be_equal_combos2(s1, s2)` thử 4 tổ hợp phép hoán đổi **không nhìn** code mẫu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `can_be_equal_n(s1, s2)` cho **độ dài bất kỳ** (LeetCode 2840). (b) Viết `min_operations(s1, s2)` trả về số thao tác **tối thiểu** (0, 1, 2) để biến `s1` thành `s2`, hoặc `-1` nếu không thể (chỉ thao tác trên `s1`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"abcd"[0::2] = "ac"`, `"abcd"[1::2] = "bd"`, `sorted("ca") = ['a', 'c']`, `swap("abcd", 1, 3) = "adcb"`. Bài 2: viết ra hai ký tự chẵn và hai ký tự lẻ của mỗi chuỗi.
- **Mức 2 (vừa):** Bài 3: năm dòng đã ghi trong comment TODO. Bài 4: `for a, b in product([0, 1], repeat=2)`, nếu `a` thì hoán đổi `(0, 2)`, nếu `b` thì hoán đổi `(1, 3)`.
- **Mức 3 (sâu):** Bài 5(a): chuỗi dài chẵn lẻ vẫn chia hai lớp theo `i % 2`; so sánh `sorted` hai lớp. Bài 5(b): thử `(a, b)` theo thứ tự tăng của `a + b` và trả `a + b` khi `t == s2`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert "abcd"[0::2] == "ac" and "abcd"[1::2] == "bd"
assert sorted("ca") == ["a", "c"]
assert swap("abcd", 1, 3) == "adcb"

# Bài 2: s1 = "abcd"
assert Solution().canBeEqual("abcd", "cbad") is True      # chỉ phép A
assert Solution().canBeEqual("abcd", "dcba") is False     # d ở vị trí 0 (chẵn) mà d ở vị trí 3 (lẻ) của s1
assert Solution().canBeEqual("abcd", "badc") is False     # b, a bị đổi tính chẵn lẻ

# Bài 3
def can_be_equal_todo_ans(s1, s2):
    even1, even2 = s1[0::2], s2[0::2]
    odd1, odd2 = s1[1::2], s2[1::2]
    return sorted(even1) == sorted(even2) and sorted(odd1) == sorted(odd2)

assert can_be_equal_todo_ans("abcd", "cdab") is True
assert can_be_equal_todo_ans("abcd", "dacb") is False

# Bài 4
def can_be_equal_combos2(s1, s2):
    for a, b in product([0, 1], repeat=2):
        t = s1
        if a:
            t = swap(t, 0, 2)
        if b:
            t = swap(t, 1, 3)
        if t == s2:
            return True
    return False

assert can_be_equal_combos2("abcd", "cdab") is True
assert can_be_equal_combos2("abcd", "dacb") is False

# Bài 5(a)
def can_be_equal_n(s1, s2):
    if len(s1) != len(s2):
        return False
    return sorted(s1[0::2]) == sorted(s2[0::2]) and sorted(s1[1::2]) == sorted(s2[1::2])

assert can_be_equal_n("abcde", "cbade") is True      # hoán đổi (0, 2)
assert can_be_equal_n("abcdef", "cdefab") is True    # lớp chẵn ace <-> cea, lớp lẻ bdf <-> dfb
assert can_be_equal_n("abcdef", "bacdef") is False   # b nằm ở vị trí chẵn của s2 nhưng lẻ của s1

# BFS tổng quát cho độ dài n (hoán đổi mọi cặp i, i+2) để làm oracle
def reachable_n(s):
    seen = {s}
    queue = deque([s])
    while queue:
        cur = queue.popleft()
        for i in range(len(cur) - 2):
            nxt = swap(cur, i, i + 2)
            if nxt not in seen:
                seen.add(nxt)
                queue.append(nxt)
    return seen

strings6 = ["".join(p) for p in product("ab", repeat=6)]
for s1 in strings6:
    reach = reachable_n(s1)
    for s2 in strings6:
        assert can_be_equal_n(s1, s2) == (s2 in reach), (s1, s2)
print("Bài 5(a): công thức tổng quát khớp BFS trên mọi chuỗi 'ab' độ dài 6")

# Bài 5(b)
def min_operations(s1, s2):
    best = -1
    for a, b in product([0, 1], repeat=2):
        t = s1
        if a:
            t = swap(t, 0, 2)
        if b:
            t = swap(t, 1, 3)
        if t == s2 and (best == -1 or a + b < best):
            best = a + b
    return best

assert min_operations("abcd", "abcd") == 0
assert min_operations("abcd", "cbad") == 1
assert min_operations("abcd", "cdab") == 2
assert min_operations("abcd", "dacb") == -1

random.seed(3)
for _ in range(300):
    s1 = "".join(random.choice("abc") for _ in range(4))
    s2 = "".join(random.choice("abc") for _ in range(4))
    assert can_be_equal_todo_ans(s1, s2) == can_be_equal_combos2(s1, s2) == can_be_equal_n(s1, s2)
    assert (min_operations(s1, s2) != -1) == Solution().canBeEqual(s1, s2)
print("Tất cả đáp án đúng")

Bài 5(a): công thức tổng quát khớp BFS trên mọi chuỗi 'ab' độ dài 6
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hoán đổi khoảng cách 2 ⇒ so sánh đa tập theo lớp chẵn / lẻ)

```python
class Solution:
    def canBeEqual(self, s1: str, s2: str) -> bool:
        return (sorted(s1[0::2]) == sorted(s2[0::2])
                and sorted(s1[1::2]) == sorted(s2[1::2]))
```

**Checklist**

- [ ] Hiểu chỉ có hai phép hoán đổi `(0, 2)` và `(1, 3)`, nên chẵn / lẻ **độc lập**?
- [ ] Không so sánh `sorted` của cả chuỗi (sai ở ví dụ 2)?
- [ ] Không hoán đổi các vị trí kề nhau?
- [ ] Nhớ mỗi phép dùng 0 hoặc 1 lần là đủ (nếu cài đặt bằng thử tổ hợp)?
- [ ] Đã thử: giống hệt, chỉ A, chỉ B, cả A và B, ký tự lặp?
- [ ] Đã vét cạn mọi cặp trên bảng chữ nhỏ so với BFS?
- [ ] Signature đúng `canBeEqual(self, s1: str, s2: str) -> bool`?